# 01 · Transcription Pipeline

Run [comic-transcriber](https://github.com/nearestnabors/comic-transcriber) on each page to extract structured panel descriptions and dialogue, then parse and save as JSON for downstream embedding pipelines.

**Output per page:**
```json
{
  "page_id": "action_comics_001_p003",
  "comic_title": "Action Comics 001",
  "page_num": 3,
  "file_path": "data/pages/...",
  "all_dialogue": "Look out! The villains are here...",
  "scene_descriptions": "A hero leaps across rooftops...",
  "raw_transcript": "## PAGE 3, PANEL 1..."
}
```

In [ ]:
import json
import re
import subprocess
from pathlib import Path

import pandas as pd
from tqdm import tqdm

TRANSCRIPTS_DIR = Path("data/transcripts")
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv("data/manifest.csv")
print(f"Pages to transcribe: {len(manifest)}")

## Install comic-transcriber

comic-transcriber runs via `npx`. Node.js must be installed.

In [ ]:
# Verify npx is available
result = subprocess.run(["npx", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("npx not found — install Node.js from https://nodejs.org")
print(f"npx {result.stdout.strip()} ✓")

## Parsing helpers

In [ ]:
def parse_transcript(raw: str) -> tuple[str, str]:
    """Extract dialogue and scene descriptions from comic-transcriber markdown output."""
    dialogue_lines = []
    description_lines = []

    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        # Italic lines (wrapped in * or _) are scene descriptions
        if re.match(r'^[*_].+[*_]$', line):
            description_lines.append(re.sub(r'^[*_]|[*_]$', '', line))
        # Bold name followed by colon is dialogue: **CHARACTER**: text
        elif re.match(r'^\*\*.+\*\*:', line):
            dialogue_lines.append(re.sub(r'^\*\*.+\*\*:\s*', '', line))
        # Plain lines without markup — treat as description
        elif not line.startswith("["):
            description_lines.append(line)

    return " ".join(dialogue_lines), " ".join(description_lines)

## Run transcription

In [ ]:
def transcribe_page(image_path: str) -> str:
    """Run comic-transcriber on a single page image and return raw markdown."""
    result = subprocess.run(
        ["npx", "comic-transcriber", image_path],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode != 0:
        raise RuntimeError(f"comic-transcriber failed: {result.stderr[:500]}")
    return result.stdout


records = []
errors = []

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Transcribing"):
    out_file = TRANSCRIPTS_DIR / f"{row['page_id']}.json"
    if out_file.exists():
        records.append(json.loads(out_file.read_text()))
        continue
    try:
        raw = transcribe_page(row["file_path"])
        dialogue, descriptions = parse_transcript(raw)
        record = {
            "page_id":           row["page_id"],
            "comic_title":       row["comic_title"],
            "comic_slug":        row["comic_slug"],
            "page_num":          int(row["page_num"]),
            "file_path":         row["file_path"],
            "all_dialogue":      dialogue,
            "scene_descriptions": descriptions,
            "raw_transcript":    raw,
        }
        out_file.write_text(json.dumps(record, ensure_ascii=False, indent=2))
        records.append(record)
    except Exception as e:
        errors.append({"page_id": row["page_id"], "error": str(e)})
        print(f"  ✗ {row['page_id']}: {e}")

print(f"\n✓ {len(records)} transcribed  ✗ {len(errors)} errors")

## Spot-check

In [ ]:
if records:
    sample = records[0]
    print(f"page_id:          {sample['page_id']}")
    print(f"all_dialogue:     {sample['all_dialogue'][:300]}")
    print(f"scene_descriptions: {sample['scene_descriptions'][:300]}")
    print(f"\nraw_transcript preview:\n{sample['raw_transcript'][:500]}")

In [ ]:
# Summary stats
df = pd.DataFrame(records)
print(df[["comic_title", "page_num", "all_dialogue", "scene_descriptions"]]
      .assign(dialogue_len=df["all_dialogue"].str.len(),
              desc_len=df["scene_descriptions"].str.len())
      .groupby("comic_title")[["dialogue_len", "desc_len"]]
      .describe())